In [1]:
from lexico import *
from sintactico_ast import *
import json

In [2]:
codigo_fuente = """
inicio
    a = 10
    b = 20
    c = a + b * 2
    si (c > 30) entonces
        escribir(c)
        d = c - 10
    finsi
    escribir(d)
fin
"""

print("--- 1. CÓDIGO FUENTE ---")
print(codigo_fuente)

print("\n--- 2. TOKENS GENERADOS (Léxico) ---")
tokens = identificar_tokens(codigo_fuente)
parser = Parser(tokens)
arbol = parser.parsear()
for token in tokens:
    print(token)
    

--- 1. CÓDIGO FUENTE ---

inicio
    a = 10
    b = 20
    c = a + b * 2
    si (c > 30) entonces
        escribir(c)
        d = c - 10
    finsi
    escribir(d)
fin


--- 2. TOKENS GENERADOS (Léxico) ---
('KEYWORD', 'inicio')
('IDENTIFIER', 'a')
('OPERATOR', '=')
('NUMBER', '10')
('IDENTIFIER', 'b')
('OPERATOR', '=')
('NUMBER', '20')
('IDENTIFIER', 'c')
('OPERATOR', '=')
('IDENTIFIER', 'a')
('OPERATOR', '+')
('IDENTIFIER', 'b')
('OPERATOR', '*')
('NUMBER', '2')
('KEYWORD', 'si')
('DELIMITER', '(')
('IDENTIFIER', 'c')
('OPERATOR', '>')
('NUMBER', '30')
('DELIMITER', ')')
('KEYWORD', 'entonces')
('KEYWORD', 'escribir')
('DELIMITER', '(')
('IDENTIFIER', 'c')
('DELIMITER', ')')
('IDENTIFIER', 'd')
('OPERATOR', '=')
('IDENTIFIER', 'c')
('OPERATOR', '-')
('NUMBER', '10')
('KEYWORD', 'finsi')
('KEYWORD', 'escribir')
('DELIMITER', '(')
('IDENTIFIER', 'd')
('DELIMITER', ')')
('KEYWORD', 'fin')


In [3]:
def imprimir_ast(nodo):
    # 0. Nodo Raíz (Programa)
    if isinstance(nodo, NodoPrograma):
        return {
            "Tipo": "Programa",
            "Funciones": [imprimir_ast(f) for f in nodo.funcion]
        }
        
    # 1. Nodos Principales
    elif isinstance(nodo, NodoFuncion):
        return {
            "Tipo": "Funcion",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Retorno": nodo.tipo[1] if isinstance(nodo.tipo, tuple) else nodo.tipo,
            "Parametros": [imprimir_ast(p) for p in nodo.parametros],
            "Cuerpo": [imprimir_ast(c) for c in nodo.cuerpo]
        }
    
    # 2. Llamadas y Prints
    elif isinstance(nodo, NodoLlamada):
        return {
            "Tipo": "LlamadaFuncion",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Argumentos": [imprimir_ast(a) for a in nodo.argumentos]
        }
    elif isinstance(nodo, NodoPrint):
        return {
            "Tipo": "SentenciaPrint",
            "Expresiones": [imprimir_ast(e) for e in nodo.expresiones]
        }
    elif isinstance(nodo, NodoString):
        return {
            "Tipo": "String",
            "Valor": nodo.valor[1] if isinstance(nodo.valor, tuple) else nodo.valor
        }

    # 3. Nodos Base
    elif isinstance(nodo, NodoParametros):
        return {
            "Tipo": "Parametro",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Dato": nodo.tipo[1] if isinstance(nodo.tipo, tuple) else nodo.tipo
        }
    elif isinstance(nodo, NodoAsignacion):
        return {
            "Tipo": "Asignacion",
            "Variable": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Expresion": imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            "Tipo": "Operacion",
            "Operador": nodo.operador[1] if isinstance(nodo.operador, tuple) else nodo.operador,
            "Izquierda": imprimir_ast(nodo.izquierda),
            "Derecha": imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoRetorno):
        return {
            "Tipo": "Retorno",
            "Valor": imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoNumero):
        return {"Numero": nodo.valor[1] if isinstance(nodo.valor, tuple) else nodo.valor}
    elif isinstance(nodo, NodoIdentificador):
        return {"Identificador": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre}
    elif isinstance(nodo, NodoSi):
        return {
            "Tipo": "Si",
            "Condicion": imprimir_ast(nodo.condicion),
            "Cuerpo": [imprimir_ast(instruccion) for instruccion in nodo.cuerpo]
        }
    
    return {"Error": "Nodo desconocido"}

In [4]:
# 1. Tokenizar y Parsear
tokens = identificar_tokens(codigo_fuente)
parser = Parser(tokens)
arbol_ast = parser.parsear()

# 2. Convertir a JSON
ast_json = json.dumps(imprimir_ast(arbol_ast), indent=2)

# 3. Imprimir para copiar
print(ast_json)

{
  "Tipo": "Programa",
  "Funciones": [
    {
      "Tipo": "Funcion",
      "Nombre": "inicio",
      "Retorno": "void",
      "Parametros": [],
      "Cuerpo": [
        {
          "Tipo": "Asignacion",
          "Variable": "a",
          "Expresion": {
            "Numero": "10"
          }
        },
        {
          "Tipo": "Asignacion",
          "Variable": "b",
          "Expresion": {
            "Numero": "20"
          }
        },
        {
          "Tipo": "Asignacion",
          "Variable": "c",
          "Expresion": {
            "Tipo": "Operacion",
            "Operador": "*",
            "Izquierda": {
              "Tipo": "Operacion",
              "Operador": "+",
              "Izquierda": {
                "Identificador": "a"
              },
              "Derecha": {
                "Identificador": "b"
              }
            },
            "Derecha": {
              "Numero": "2"
            }
          }
        },
        {
          "Tipo": "S

In [5]:
codigo = arbol_ast.generarCodigo()
print(codigo)

inicio:

    mov eax, 10
    mov [a], eax

    mov eax, 20
    mov [b], eax

    mov eax [a]
    push    eax

    mov eax [b]
    mov    ebx, eax
    pop    eax
    add    eax, ebx
    push    eax

    mov eax, 2
    mov    ebx, eax
    pop    eax
    mov [c], eax
; Instruccion IF no implementada en ASM aun

    mov eax [d]
    call print_int
    call print_newline
    ret

section .bss

section .text
global _start
_start:
    call main
    mov eax, 1    ; syscall exit
    xor ebx, ebx  ; Codigo de Salida 0
    int 0x80


In [6]:
import semantico

In [7]:
analizador_semantico = semantico.AnalizadorSemantico()
analisis = analizador_semantico.tabla_simbolos

In [8]:
print("\n ----- Semántico/Optimización ----- ")
arbol = arbol.optimizar()
arbol.generarCodigo()


 ----- Semántico/Optimización ----- 
inicio:

    mov eax, 10
    mov [a], eax

    mov eax, 20
    mov [b], eax

    mov eax, 60
    mov [c], eax

    mov eax, 60
    call print_int
    call print_newline

    mov eax, 50
    mov [d], eax

    mov eax, 50
    call print_int
    call print_newline
    ret



'section .bss\n\nsection .text\nglobal _start\n_start:\n    call main\n    mov eax, 1    ; syscall exit\n    xor ebx, ebx  ; Codigo de Salida 0\n    int 0x80'

In [9]:
import subprocess

In [10]:
def compilar(programa):
    #Generar el codifo en ensamblador
    codigo_asm = programa.generarCodigo()
    print(codigo_asm)
    text_file = open("ejasmcompiladores.asm","w")
    text_file.write(codigo_asm)
    text_file.close()
    subprocess.run(["nasm", "-f", "elf", "ejasmcompiladores.asm"])
    subprocess.run(["ld", "-m", "elf_i386", "ejasmcompiladores.o", "-o", "ejasmcompiladores"])


In [11]:
#compilar(arbol_ast)  #Al ejecutar quitar el comentario

In [12]:
# ── Importar el complemento semántico jerárquico ──
from semantico import (
    ejecutar_simulacion_laberinto,
    generar_c3d_bloque_interno,
    explicar_shadowing,
    imprimir_resumen_final,
)

In [13]:
# ── Parte 1 y 2: Simulación + detección de errores ──
tabla = ejecutar_simulacion_laberinto()


  SIMULACION: El Laberinto de los Ambitos

  MOMENTO A -- despues de declarar 'int y = a * 2;'
  [GLOBAL]  {'x': 'int'}
  [Nivel 1]  {'a': 'int', 'y': 'int'}
  Profundidad: 2 ambito(s) activos

  MOMENTO B -- antes de cerrar el bloque interno
  [GLOBAL]  {'x': 'int'}
  [Nivel 1]  {'a': 'int', 'y': 'int'}
  [Nivel 2]  {'x': 'float'}
  Profundidad: 3 ambito(s) activos
  Shadowing activo: 'x' visible = 'float' (float local oculta al int global)

  MOMENTO C -- al llegar a 'escribir(z);'
  [GLOBAL]  {'x': 'int'}
  [Nivel 1]  {'a': 'int', 'y': 'int'}
  Profundidad: 2 ambito(s) activos

  ERRORES DETECTADOS
  - Error Semantico [1] -- Tipos incompatibles: 'y'(int) + 'x'(float) en 'y = y + x'
  - Error Semantico [2] -- Error Semantico: 'z' no declarada en ningun ambito visible
  - Error Semantico [3] -- Shadowing: 'float x' en el bloque interno oculta a 'int x' global, bloqueando acceso a ella dentro del bloque.


In [14]:
# ── Parte 3: Shadowing ──
explicar_shadowing()


  FENOMENO DEL SHADOWING -- Pregunta 3

  Pila en MOMENTO B:
    [GLOBAL]  -> { x: int  }
    [Nivel 1] -> { a: int, y: int }
    [Nivel 2] -> { x: float }        <- tope

  obtener_tipo_variable('x') recorre de adentro -> afuera:
    1. Busca en Nivel 2 -> ENCONTRADA: float
    2. Se detiene. No llega al global.

  En  y = y + x:
      y  ->  int
      x  ->  float   (la local, no la global)
      int + float   ->  ERROR SEMANTICO



In [15]:
# ── Parte 4: Código de Tres Direcciones ──
generar_c3d_bloque_interno()


  CODIGO DE TRES DIRECCIONES -- Bloque interno
  // Inicio bloque interno { }
  x_local      = 5.5                # float x = 5.5  (shadowing -> x_local)
  t1           = y + x_local        # y(int) + x_local(float) -> ERROR tipo
  y            = t1                 
  // Fin bloque interno }
  // De vuelta en ambito de test()
  t2           = y + 1              
  x_global     = t2                 # x = y + 1  (x global int)
  x_local  -> float x del bloque interno
  x_global -> int   x del ambito global
  t1, t2   -> temporales unicos


In [16]:
# ── Parte 5: Resumen final (verifica momentos A, B, C) ──
imprimir_resumen_final(tabla)


  RESUMEN FINAL -- Historial de Pila de Ambitos
  MOMENTO A: despues de 'int y = a * 2'
     [GLOBAL] {'x': 'int'}
     [Nivel 1] {'a': 'int', 'y': 'int'}
     -> COINCIDE
  --------------------------------------------------
  MOMENTO B: antes de cerrar el bloque interno
     [GLOBAL] {'x': 'int'}
     [Nivel 1] {'a': 'int', 'y': 'int'}
     [Nivel 2] {'x': 'float'}
     -> COINCIDE
  --------------------------------------------------
  MOMENTO C: al llegar a 'escribir(z)'
     [GLOBAL] {'x': 'int'}
     [Nivel 1] {'a': 'int', 'y': 'int'}
     -> COINCIDE
  --------------------------------------------------
